# 실습 세트 1. 상관분석을 활용한 반도체 생산품질 중요 요인 찾기

## 1. CSV 파일 다운로드

먼저 아래 CSV 파일을 다운로드한 뒤, 노트북 파일과 같은 폴더에 둡니다.

파일명:

`semicond_preprocess.csv`

이 데이터는 반도체 공정 센서값과 제품의 정상/불량 여부를 담고 있는 가상 데이터입니다.

---

## 2. 문제 상황 설명

반도체 생산 공정에서는 온도, 압력, 가스 유량, 파티클 수, 두께 균일도, 전압 안정성 등 다양한 센서 데이터가 수집됩니다.

이번 실습에서는 각 생산 기록별 센서값과 제품의 불량 여부가 주어져 있습니다.

`Defect` 컬럼은 제품의 정상/불량 여부를 의미합니다.

- `Defect = 0`: 정상
- `Defect = 1`: 불량

이번 분석의 목표는 pandas와 seaborn을 사용하여 어떤 공정 센서값이 불량 여부와 관련이 큰지 탐색하는 것입니다.

단, 상관관계가 높다고 해서 반드시 그 변수가 불량의 원인이라는 뜻은 아닙니다.  
상관분석은 불량과 함께 움직이는 후보 변수를 찾는 탐색 도구로 이해해야 합니다.

### 데이터 컬럼 설명

| 컬럼명 | 설명 |
|---|---|
| Wafer_ID | 웨이퍼 ID |
| Lot_ID | 생산 Lot ID |
| Temperature | 공정 온도 |
| Chamber_Pressure | 챔버 압력 |
| Gas_Flow | 가스 유량 |
| Etch_Rate | 식각 속도 |
| Chamber_Humidity | 챔버 습도 |
| Particle_Count | 공정 중 파티클 수 |
| Thickness_Uniformity | 박막 두께 균일도 |
| Voltage_Stability | 전압 안정성 |
| Defect | 정상/불량 여부 |



# 문제 1. 데이터 기본 확인

CSV 파일을 불러온 뒤 다음 내용을 확인하세요.

- 상위 5개 행
- 데이터 크기
- 컬럼별 자료형
- 결측치 개수
- 기초 통계량

수치형 컬럼뿐 아니라 문자형 컬럼까지 함께 확인할 수 있도록 `describe(include="all")`도 사용해보세요.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("semicond_preprocess.csv")

display(df.head())

print("데이터 크기:", df.shape)

print("\n컬럼별 자료형")
display(df.dtypes)

print("\n결측치 개수")
display(df.isna().sum())

print("\n수치형 기초 통계량")
display(df.describe())

print("\n전체 컬럼 기초 통계량")
display(df.describe(include="all"))

,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
0,WAFER-00001,LOT-033,A,Day,Normal,Advanced,NaN,CH-04,479.66,2.596,120.03,1.575,33.30,20.2,0.9561,NaN,35.68,0
1,WAFER-00002,LOT-035,B,Night,Normal,Standard,S1,CH-02,458.02,2.543,117.65,1.574,32.49,14.1,0.9506,0.9740,28.96,0
2,WAFER-00003,LOT-009,A,Day,Normal,Standard,NaN,CH-02,463.43,2.372,122.07,1.559,31.98,22.4,NaN,0.9735,24.75,0
3,WAFER-00004,LOT-021,A,Day,Normal,Advanced,S2,CH-01,477.04,2.610,117.59,1.906,40.23,12.4,0.9683,0.9854,28.82,0
4,WAFER-00005,LOT-019,B,Day,High_Speed,Standard,S1,CH-02,501.19,2.193,132.00,1.520,32.20,11.9,0.9377,0.9724,29.77,0


데이터 크기: (1800, 18)

컬럼별 자료형


Wafer_ID                 object
Lot_ID                   object
Factory                  object
Shift                    object
Process_Mode             object
Product_Type             object
Material_Supplier        object
Chamber_ID               object
Temperature             float64
Chamber_Pressure        float64
Gas_Flow                float64
Etch_Rate               float64
Chamber_Humidity        float64
Particle_Count          float64
Thickness_Uniformity    float64
Voltage_Stability       float64
Cycle_Time              float64
Defect                    int64
dtype: object


결측치 개수


Wafer_ID                 0
Lot_ID                   0
Factory                  0
Shift                    0
Process_Mode            27
Product_Type             0
Material_Supplier       45
Chamber_ID               0
Temperature              0
Chamber_Pressure        63
Gas_Flow                81
Etch_Rate                0
Chamber_Humidity        54
Particle_Count           0
Thickness_Uniformity    72
Voltage_Stability       90
Cycle_Time               0
Defect                   0
dtype: int64


수치형 기초 통계량


,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
count,1800.000000,1737.000000,1719.000000,1800.000000,1746.000000,1800.000000,1728.000000,1710.000000,1800.000000,1800.000000
mean,481.689739,2.488237,119.680204,1.547538,37.837136,20.414500,0.933259,0.970479,30.955394,0.103889
std,12.506475,0.206593,7.585030,0.099642,5.482395,8.603193,0.023057,0.012109,6.780502,0.305201
min,428.750000,0.987000,93.020000,1.189000,20.270000,2.900000,0.858000,0.932400,20.510000,0.000000
25%,473.400000,2.355000,114.700000,1.481750,34.260000,16.300000,0.918000,0.962000,28.067500,0.000000
50%,481.535000,2.487000,119.690000,1.550000,37.880000,19.600000,0.933100,0.970600,30.510000,0.000000
75%,490.125000,2.621000,124.560000,1.611000,41.502500,22.900000,0.949225,0.979100,32.780000,0.000000
max,520.170000,3.414000,141.640000,1.912000,54.040000,100.900000,0.990000,1.000000,115.580000,1.000000



전체 컬럼 기초 통계량


,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
count,1800,1800,1800,1800,1773,1800,1755,1800,1800.000000,1737.000000,1719.000000,1800.000000,1746.000000,1800.000000,1728.000000,1710.000000,1800.000000,1800.000000
unique,1800,40,3,2,2,3,3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,WAFER-01800,LOT-005,A,Day,Normal,Standard,S1,CH-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,60,685,1130,1323,861,834,502,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,481.689739,2.488237,119.680204,1.547538,37.837136,20.414500,0.933259,0.970479,30.955394,0.103889
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.506475,0.206593,7.585030,0.099642,5.482395,8.603193,0.023057,0.012109,6.780502,0.305201
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,428.750000,0.987000,93.020000,1.189000,20.270000,2.900000,0.858000,0.932400,20.510000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,473.400000,2.355000,114.700000,1.481750,34.260000,16.300000,0.918000,0.962000,28.067500,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,481.535000,2.487000,119.690000,1.550000,37.880000,19.600000,0.933100,0.970600,30.510000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,490.125000,2.621000,124.560000,1.611000,41.502500,22.900000,0.949225,0.979100,32.780000,0.000000


# 문제 2. 조건에 맞는 데이터 필터링

다음 조건에 해당하는 데이터를 각각 필터링하세요.

1. 불량 제품만 선택하세요.
2. 야간 교대조(`Night`)에서 생산된 제품만 선택하세요.
3. `Factory`가 C이고 `Product_Type`이 Premium인 제품만 선택하세요.

각 필터링 결과의 데이터 개수도 함께 출력하세요.

In [2]:
defect_only = df[df["Defect"] == 1]
display(defect_only)

night_only = df[df["Shift"] == "Night"]
display(night_only)

factory_c_premium = df[
    (df["Factory"] == "C") &
    (df["Product_Type"] == "Premium")
]
display(factory_c_premium)


,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
14,WAFER-00015,LOT-033,A,Day,Normal,Premium,S2,CH-04,485.90,NaN,113.77,1.575,48.78,12.6,0.8952,0.9655,34.27,1
20,WAFER-00021,LOT-017,A,Night,High_Speed,Standard,S1,CH-04,475.81,2.458,128.66,1.613,31.62,85.0,0.9477,0.9681,34.29,1
39,WAFER-00040,LOT-005,A,Day,High_Speed,Advanced,S2,CH-01,470.32,1.392,116.43,1.677,38.47,20.4,0.9163,0.9956,24.95,1
42,WAFER-00043,LOT-007,C,Night,Normal,Advanced,S1,CH-04,478.13,2.785,131.35,1.598,NaN,29.3,0.9396,0.9578,31.01,1
52,WAFER-00053,LOT-029,B,Day,High_Speed,Advanced,S1,CH-03,495.92,2.700,118.32,1.749,30.15,16.6,0.9053,0.9704,23.93,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754,WAFER-01755,LOT-035,B,Day,Normal,Standard,S1,CH-01,467.06,2.650,126.11,1.530,34.54,16.5,0.9364,0.9841,31.51,1
1763,WAFER-01764,LOT-020,B,Day,Normal,Premium,S1,CH-04,466.34,2.481,NaN,1.534,37.82,20.3,0.8890,0.9707,34.77,1
1767,WAFER-01768,LOT-027,C,Night,High_Speed,Advanced,S3,CH-02,479.54,NaN,122.69,1.596,41.22,22.4,0.9480,0.9574,25.31,1
1783,WAFER-01784,LOT-003,A,Night,Normal,Standard,S3,CH-04,460.95,2.201,125.74,1.550,45.84,16.9,0.9314,0.9485,30.72,1


,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
1,WAFER-00002,LOT-035,B,Night,Normal,Standard,S1,CH-02,458.02,2.543,117.65,1.574,32.49,14.1,0.9506,0.9740,28.96,0
5,WAFER-00006,LOT-002,C,Night,Normal,Standard,S1,CH-03,480.79,2.214,115.73,1.599,48.44,25.4,0.9489,0.9757,30.10,0
10,WAFER-00011,LOT-025,B,Night,Normal,Advanced,S1,CH-02,512.93,2.277,134.35,1.522,35.44,22.2,0.8998,0.9528,27.45,0
11,WAFER-00012,LOT-004,A,Night,High_Speed,Advanced,S1,CH-04,467.51,2.224,118.79,1.516,40.81,19.9,0.9427,0.9691,24.13,0
18,WAFER-00019,LOT-019,C,Night,Normal,Premium,S3,CH-01,469.13,2.495,109.61,1.467,36.61,17.7,0.9191,0.9715,27.65,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785,WAFER-01786,LOT-005,C,Night,Normal,Advanced,S2,CH-04,470.84,2.741,123.09,1.401,47.54,31.7,0.9350,0.9636,30.38,0
1787,WAFER-01788,LOT-023,A,Night,Normal,Premium,S1,CH-01,494.51,2.144,93.02,1.365,32.80,18.6,0.9450,0.9517,30.96,0
1788,WAFER-01789,LOT-027,A,Night,High_Speed,Premium,S2,CH-01,474.04,2.548,115.90,1.631,32.95,19.7,0.9177,0.9589,21.89,0
1793,WAFER-01794,LOT-010,B,Night,Normal,Advanced,S2,CH-01,492.04,2.625,103.06,1.481,39.89,16.9,0.9190,0.9655,32.76,0


,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
18,WAFER-00019,LOT-019,C,Night,Normal,Premium,S3,CH-01,469.13,2.495,109.61,1.467,36.61,17.7,0.9191,0.9715,27.65,0
29,WAFER-00030,LOT-033,C,Day,Normal,Premium,S1,CH-04,481.52,2.372,122.54,1.576,38.50,22.6,0.9132,0.9760,33.20,0
58,WAFER-00059,LOT-006,C,Day,Normal,Premium,S2,CH-01,482.37,2.231,117.65,1.509,30.48,14.9,0.9133,0.9750,32.54,0
76,WAFER-00077,LOT-024,C,Day,Normal,Premium,S1,CH-03,493.46,2.408,117.16,1.646,45.64,25.2,0.9518,0.9648,32.02,0
81,WAFER-00082,LOT-009,C,Day,High_Speed,Premium,S3,CH-03,462.46,2.730,125.11,1.439,NaN,25.9,0.9115,0.9883,25.58,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1705,WAFER-01706,LOT-009,C,Day,NaN,Premium,S2,CH-01,487.91,NaN,120.15,1.559,37.19,25.2,0.9198,0.9497,30.12,0
1714,WAFER-01715,LOT-007,C,Day,Normal,Premium,S2,CH-01,474.51,2.575,121.53,1.604,32.35,14.5,0.8966,0.9515,29.25,0
1758,WAFER-01759,LOT-022,C,Day,High_Speed,Premium,S1,CH-03,481.29,2.569,125.11,1.409,42.83,16.6,NaN,0.9741,30.46,0
1773,WAFER-01774,LOT-017,C,Day,Normal,Premium,S2,CH-04,458.05,2.412,117.85,1.692,38.92,15.8,0.9146,0.9686,33.90,0


# 문제 3. 상관계수를 구하여 변수 선택하기

수치형 컬럼만 선택한 뒤 상관계수를 계산하세요.

그다음 `Defect`와 다른 수치형 변수들의 상관계수를 구하고, 절댓값 기준으로 정렬하세요.

`Defect`와 관련이 높은 상위 5개 변수를 선택하세요.


In [3]:
numeric_df = df.select_dtypes(include="number")

corr_matrix = numeric_df.corr()

defect_corr = corr_matrix["Defect"].drop("Defect")
    # .sort_values(key=lambda s: s.abs(), ascending=False)

defect_corr = pd.DataFrame(defect_corr).reset_index()
defect_corr['abs'] = defect_corr['Defect'].abs()
defect_corr = defect_corr.sort_values('abs', ascending=False)
display(defect_corr)

top5_features = defect_corr.head(5)['index'].tolist()

print("Defect와 상관성이 높은 상위 5개 변수:")
for col in top5_features:
    print("-", col)

,index,Defect,abs
5,Particle_Count,0.279464,0.279464
6,Thickness_Uniformity,-0.172961,0.172961
7,Voltage_Stability,-0.121107,0.121107
2,Gas_Flow,-0.071142,0.071142
8,Cycle_Time,0.060394,0.060394
0,Temperature,-0.038028,0.038028
1,Chamber_Pressure,-0.018410,0.018410
3,Etch_Rate,-0.015401,0.015401
4,Chamber_Humidity,-0.011717,0.011717


Defect와 상관성이 높은 상위 5개 변수:
- Particle_Count
- Thickness_Uniformity
- Voltage_Stability
- Gas_Flow
- Cycle_Time


# 문제 4. 결측치 처리

결측치가 있는 컬럼을 확인한 뒤, 다음 방식으로 결측치를 처리하세요.

- 수치형 컬럼의 결측치는 중앙값으로 채우세요.
- 문자형 컬럼의 결측치는 `"Unknown"`으로 채우세요.
- 결측치 처리 후 결측치가 남아 있는지 다시 확인하세요.

원본 데이터는 보존하고, 결측치 처리 결과는 `df_clean`이라는 새 데이터프레임에 저장하세요.

In [4]:
df_clean = df.copy()

missing_count = df_clean.isna().sum()
missing_count = missing_count[missing_count > 0]
display(missing_count)

numeric_cols = df_clean.select_dtypes(include="number").columns
category_cols = df_clean.select_dtypes(include="object").columns

median_value = df_clean[numeric_cols].median()
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(median_value)

df_clean[category_cols] = df_clean[category_cols].fillna("Unknown")

print("결측치 처리 후 결측치 개수")
display(df_clean.isna().sum())

Process_Mode            27
Material_Supplier       45
Chamber_Pressure        63
Gas_Flow                81
Chamber_Humidity        54
Thickness_Uniformity    72
Voltage_Stability       90
dtype: int64

결측치 처리 후 결측치 개수


Wafer_ID                0
Lot_ID                  0
Factory                 0
Shift                   0
Process_Mode            0
Product_Type            0
Material_Supplier       0
Chamber_ID              0
Temperature             0
Chamber_Pressure        0
Gas_Flow                0
Etch_Rate               0
Chamber_Humidity        0
Particle_Count          0
Thickness_Uniformity    0
Voltage_Stability       0
Cycle_Time              0
Defect                  0
dtype: int64

# 문제 5. IQR 방식으로 이상치 처리하기

`Particle_Count` 컬럼을 대상으로 IQR 방식의 이상치 기준을 계산하세요.

다음 값을 구하세요.

- Q1
- Q3
- IQR
- lower bound
- upper bound

그다음 `Particle_Count`가 IQR 기준을 벗어난 행을 찾으세요.


1. 이상치는 몇개인가요?
2. 이상치를 가지는 행을 삭제하세요.

In [5]:
target_col = "Particle_Count"

Q1 = df_clean[target_col].quantile(0.25)
Q3 = df_clean[target_col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("lower_bound:", lower_bound)
print("upper_bound:", upper_bound)

outlier_condition = (
    (df_clean[target_col] < lower_bound) |
    (df_clean[target_col] > upper_bound)
)

outliers = df_clean[outlier_condition]

print("이상치 개수:", outliers.shape[0])
display(outliers.head())

# 방법 1. 이상치 행 제거
df_no_outlier = df_clean[~outlier_condition].copy()

print("이상치 제거 전:", df_clean.shape)
print("이상치 제거 후:", df_no_outlier.shape)



Q1: 16.3
Q3: 22.9
IQR: 6.599999999999998
lower_bound: 6.400000000000004
upper_bound: 32.8
이상치 개수: 39


,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
20,WAFER-00021,LOT-017,A,Night,High_Speed,Standard,S1,CH-04,475.81,2.458,128.66,1.613,31.62,85.0,0.9477,0.9681,34.29,1
92,WAFER-00093,LOT-023,C,Day,Normal,Standard,S1,CH-03,490.58,2.351,117.24,1.328,37.88,35.4,0.9480,0.9679,33.35,1
173,WAFER-00174,LOT-014,C,Day,Normal,Advanced,S1,CH-04,481.15,2.399,111.46,1.427,51.92,57.6,0.9135,0.9703,31.84,1
187,WAFER-00188,LOT-026,A,Day,High_Speed,Standard,S2,CH-03,484.76,2.362,130.62,1.575,32.38,33.9,0.9332,0.9671,32.08,0
214,WAFER-00215,LOT-022,B,Day,Normal,Standard,S1,CH-01,481.03,2.487,124.91,1.596,27.29,77.5,0.9401,0.9559,28.52,1


이상치 제거 전: (1800, 18)
이상치 제거 후: (1761, 18)
